# Predicting coding status from human RefSeq transcripts

**FlyRank ML track — reproducible baseline study**

This notebook answers one deliberately narrow question:

> Can simple sequence features distinguish **coding RefSeq transcripts** from **non-coding RefSeq transcripts** in human RNA records?

The workflow verifies the source and labels, makes a simple baseline, compares two models, uses a held-out evaluation split, and inspects errors and limitations. **FLIP2 results and variant information are intentionally out of scope.**

**Run time:** the default setting downloads one current NCBI RefSeq RNA shard and uses a balanced sample of up to 1,000 records per class, so it is practical in Colab. Increase `MAX_PER_CLASS` or add shards only after the baseline is understood.

## 1. Data source and label definition

- **Source:** NCBI RefSeq human RNA FASTA directory: <https://ftp.ncbi.nlm.nih.gov/refseq/H_sapiens/mRNA_Prot/>
- **Default file:** `human.1.rna.fna.gz`
- **Label:** `1` (coding) when the RefSeq accession starts with `NM_`; `0` (non-coding) when it starts with `NR_`. Other accession types are excluded.
- **Grouping key:** gene symbol parsed from the header. All transcripts from the same parsed gene are kept in one split to reduce transcript-family leakage.

This is a **source-defined transcript-type label**, not an independent biological assay. NCBI may update the files, so the exact URL and download date are recorded at runtime.

In [ ]:
# Configuration: change these knobs after the first successful run.
import re, gzip, os, random, urllib.request
from collections import Counter
from datetime import datetime, timezone

SEED = 42
MAX_PER_CLASS = 1000        # Set to None to use every parsed record; otherwise balance classes.
NCBI_SHARDS = [1]            # Add more integers, e.g. [1, 2, 3], for a larger study.
TEST_SIZE = 0.20
DATA_DIR = "ncbi_refseq_cache"
random.seed(SEED)
os.makedirs(DATA_DIR, exist_ok=True)
print("UTC run time:", datetime.now(timezone.utc).isoformat())

In [ ]:
def parse_header(header):
    accession = header.split()[0].lstrip(">")
    if accession.startswith("NM_"):
        label, label_name = 1, "coding"
    elif accession.startswith("NR_"):
        label, label_name = 0, "non-coding"
    else:
        return None
    match = re.search(r"\(([^()]+)\)", header)
    gene = match.group(1).strip() if match else accession
    gene = re.sub(r"\s+", "_", gene)
    return {"accession": accession, "gene": gene, "label": label, "label_name": label_name}

def read_refseq_shards(shards=NCBI_SHARDS):
    rows, urls = [], []
    for shard in shards:
        url = f"https://ftp.ncbi.nlm.nih.gov/refseq/H_sapiens/mRNA_Prot/human.{shard}.rna.fna.gz"
        urls.append(url)
        local = os.path.join(DATA_DIR, url.rsplit("/", 1)[-1])
        if not os.path.exists(local):
            print("Downloading", url)
            urllib.request.urlretrieve(url, local)
        with gzip.open(local, "rt") as fh:
            header, seq_parts = None, []
            for line in fh:
                line = line.strip()
                if line.startswith(">"):
                    if header is not None:
                        meta = parse_header(header)
                        if meta:
                            meta["sequence"] = "".join(seq_parts).upper()
                            rows.append(meta)
                    header, seq_parts = line, []
                else:
                    seq_parts.append(line)
            if header is not None:
                meta = parse_header(header)
                if meta:
                    meta["sequence"] = "".join(seq_parts).upper()
                    rows.append(meta)
    if MAX_PER_CLASS is None:
        return rows, urls
    rng = random.Random(SEED)
    by_label = {0: [r for r in rows if r["label"] == 0], 1: [r for r in rows if r["label"] == 1]}
    n = min(MAX_PER_CLASS, len(by_label[0]), len(by_label[1]))
    sampled = []
    for label in (0, 1):
        rng.shuffle(by_label[label])
        sampled.extend(by_label[label][:n])
    rng.shuffle(sampled)
    return sampled, urls

records, source_urls = read_refseq_shards()
print(f"Records loaded after class-balanced sampling: {len(records):,}")
print("Labels:", Counter(r["label_name"] for r in records))
print("Unique genes:", len({r["gene"] for r in records}))
print("Example:", records[0]["accession"], records[0]["gene"], records[0]["label_name"], len(records[0]["sequence"]))

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = pd.DataFrame(records)
df["length"] = df["sequence"].str.len()
df["gc_fraction"] = df["sequence"].map(lambda s: (s.count("G") + s.count("C")) / max(1, len(s)))
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(gss.split(df, y=df["label"], groups=df["gene"]))
train = df.iloc[train_idx].reset_index(drop=True)
test = df.iloc[test_idx].reset_index(drop=True)
print(f"Train: {len(train):,} records / {train.gene.nunique():,} genes")
print(f"Test:  {len(test):,} records / {test.gene.nunique():,} genes")
print("Gene overlap:", len(set(train.gene) & set(test.gene)))
print("Train label rate:", train.label.mean().round(3), "| Test label rate:", test.label.mean().round(3))
train.head()

## 2. Baselines and models

1. **Majority-class baseline:** always predicts the most common training label.
2. **Composition baseline:** logistic regression using transcript length and GC fraction.
3. **Sequence model:** character 3–5-mer TF–IDF features with logistic regression. The vectorizer is fit on training sequences only.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, classification_report, confusion_matrix
import numpy as np

def evaluate(name, y_true, y_pred, y_score=None):
    row = {"model": name, "accuracy": accuracy_score(y_true, y_pred), "balanced_accuracy": balanced_accuracy_score(y_true, y_pred)}
    if y_score is not None and len(np.unique(y_true)) == 2:
        row["roc_auc"] = roc_auc_score(y_true, y_score)
    return row

results = []
majority = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(train), 1)), train.label)
pred = majority.predict(np.zeros((len(test), 1)))
results.append(evaluate("majority baseline", test.label, pred))

composition = Pipeline([("scale", StandardScaler()), ("logreg", LogisticRegression(max_iter=1000, random_state=SEED))])
composition.fit(train[["length", "gc_fraction"]], train.label)
pred = composition.predict(test[["length", "gc_fraction"]])
score = composition.predict_proba(test[["length", "gc_fraction"]])[:, 1]
results.append(evaluate("length + GC logistic regression", test.label, pred, score))

sequence_model = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=2, max_features=50000, sublinear_tf=True)),
    ("logreg", LogisticRegression(max_iter=1000, C=2.0, class_weight="balanced", random_state=SEED))
])
sequence_model.fit(train.sequence, train.label)
pred = sequence_model.predict(test.sequence)
score = sequence_model.predict_proba(test.sequence)[:, 1]
results.append(evaluate("character 3-5mer TF-IDF + logistic regression", test.label, pred, score))

results_df = pd.DataFrame(results).sort_values("balanced_accuracy", ascending=False)
results_df

In [ ]:
seq_pred = sequence_model.predict(test.sequence)
seq_score = sequence_model.predict_proba(test.sequence)[:, 1]
print(classification_report(test.label, seq_pred, target_names=["non-coding", "coding"], digits=3))
print("Confusion matrix [rows=true, cols=predicted]:\n", confusion_matrix(test.label, seq_pred))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=results_df, x="balanced_accuracy", y="model", ax=axes[0], color="#3568a8")
axes[0].set_xlim(0, 1); axes[0].set_title("Held-out balanced accuracy")
cm = confusion_matrix(test.label, seq_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1])
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True"); axes[1].set_title("Sequence model confusion matrix")
plt.tight_layout(); plt.show()

In [ ]:
error_view = test[["accession", "gene", "label_name", "length", "gc_fraction"]].copy()
error_view["predicted_label"] = np.where(seq_pred == 1, "coding", "non-coding")
error_view["coding_probability"] = seq_score
error_view["correct"] = error_view.label_name == error_view.predicted_label
error_view["uncertainty"] = (error_view.coding_probability - 0.5).abs()
error_view.sort_values("uncertainty").head(20)

## 3. Interpretation, limitations, and next steps

- **Leakage control:** the split is by parsed gene, and the notebook prints the gene-overlap count. It should be zero. This is stronger than a random transcript split, but it is not a full sequence-similarity clustering analysis.
- **Label limitations:** `NM_`/`NR_` is a RefSeq accession convention used here as the operational label; it can include edge cases and does not replace expert curation.
- **Dataset shift:** the default shard is only a reproducible slice of human RefSeq and may be updated by NCBI. Report the run timestamp, URLs, and record cap with any result.
- **Confounding:** sequence length, GC composition, transcript family, and annotation practices may drive performance. The composition baseline helps reveal whether k-mers add signal beyond simple composition.
- **Not included:** FLIP2 results and variant information are deliberately separate from this study.

### Suggested researcher review

Ask a bioinformatics researcher to review (1) the `NM_`/`NR_` label definition, (2) whether gene-symbol grouping is sufficient for leakage prevention, (3) whether the selected RefSeq slice is representative, and (4) whether a stronger external test set is needed.

In [ ]:
manifest = {
    "seed": SEED,
    "max_per_class": MAX_PER_CLASS,
    "ncbi_shards": NCBI_SHARDS,
    "test_size": TEST_SIZE,
    "source_urls": source_urls,
    "download_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "records_loaded": len(df),
    "train_records": len(train),
    "test_records": len(test),
    "gene_overlap": len(set(train.gene) & set(test.gene)),
}
manifest